In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer, make_column_selector

In [15]:
sticker = pd.read_csv('D:/CDAC/ACTS-Pune/Subjects_And_StudyMaterial/Mine/MachineLearning/Lab_assignments/A1/train.csv')
sticker.head()

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0


In [16]:
sticker["num_sold"].isna().sum()
sticker["num_sold"].fillna(sticker["num_sold"].mean(), inplace=True)

C:\Users\Vivek Gotecha\AppData\Local\Temp\ipykernel_22856\1456225660.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sticker["num_sold"].fillna(sticker["num_sold"].mean(), inplace=True)


In [17]:
sticker.shape

(230130, 6)

In [18]:
sticker.info()
sticker.drop('id', axis=1, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230130 entries, 0 to 230129
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   id        230130 non-null  int64  
 1   date      230130 non-null  object 
 2   country   230130 non-null  object 
 3   store     230130 non-null  object 
 4   product   230130 non-null  object 
 5   num_sold  230130 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 10.5+ MB


In [19]:
X = sticker.drop('num_sold', axis=1)
y = sticker['num_sold']


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [21]:
ohe = OneHotEncoder(sparse_output=True, drop = 'first')

In [22]:
ohe = OneHotEncoder(sparse_output=True, drop = 'first')

In [23]:
transformer_1 = ColumnTransformer(transformers=[('OHE', ohe, make_column_selector(dtype_include=object))], remainder='passthrough', verbose_feature_names_out=False)

In [24]:
X_train_trans = transformer_1.fit_transform(X_train)
X_test_trans = transformer_1.transform(X_test)

In [25]:
model = LinearRegression()
model.fit(X_train_trans,y_train)

LinearRegression()

In [26]:
y_pred = model.predict(X_test_trans)
r2_score(y_test, y_pred)

0.6914132351144311

In [27]:
alpha_values = np.linspace(0.01, 10)
dict_alpha = dict()
for i in alpha_values:
    lasso = Lasso(alpha = i)
    lasso.fit(X_train_trans, y_train)
    y_pred_lasso = lasso.predict(X_test_trans)
    dict_alpha[i] = r2_score(y_test, y_pred_lasso)
    
lasso_df = pd.DataFrame(dict_alpha.items(), columns = ['Alpha', 'Score'])
lasso_df.sort_values('Score', ascending = False).iloc[0]

Alpha    0.010000
Score    0.692406
Name: 0, dtype: float64

In [28]:
alpha_values = np.linspace(0.01, 10)
dict_alpha = dict()
for i in alpha_values:
    ridge = Ridge(alpha = i)
    ridge.fit(X_train_trans, y_train)
    y_pred_ridge = ridge.predict(X_test_trans)
    dict_alpha[i] = r2_score(y_test, y_pred_ridge)
    
ridge_df = pd.DataFrame(dict_alpha.items(), columns = ['Alpha', 'Score'])
ridge_df.sort_values('Score', ascending = False).iloc[0]

Alpha    10.000000
Score     0.692421
Name: 49, dtype: float64

In [ ]:
alpha_values = np.linspace(0.01, 10)
l1_ratio = np.linspace(0, 0.5)
scores=[]
dict_alpha = dict()
for i,j in zip(alpha_values,l1_ratio):
    ridge = ElasticNet(alpha = i,l1_ratio=j)
    ridge.fit(X_train_trans, y_train)
    y_pred_ridge = ridge.predict(X_test_trans)
    # dict_alpha[i] = r2_score(y_test, y_pred_ridge)
    scores.append([i,j, r2_score(y_test, y_pred_ElasticNet)])

elastic_df = pd.DataFrame(scores, columns = ['Alpha', 'L1_Ratio', 'Score'])
elastic_df.sort_values('Score', ascending = False).iloc[0]